# 🏥 MedAssist — Fine-tuning no Google Colab
### Tech Challenge IADT — Fase 3 | POSTECH / FIAP

Este notebook realiza o **fine-tuning completo** de um LLM usando:
- **Modelo base:** `unsloth/Phi-3-mini-4k-instruct` (3.8B, eficiente para T4)
- **Técnica:** QLoRA (4-bit) via `unsloth` (2x mais rápido que HuggingFace puro)
- **Dataset:** MedQuAD PT-BR (5.274 pares pergunta/resposta médica)

---
## ⚙️ Pré-requisitos
1. Vá em **Runtime → Change runtime type → T4 GPU**
2. Execute as células em ordem
3. Na **Célula 3**, faça upload do arquivo `medquad_ptbr.jsonl`
4. Ao final, o modelo será salvo no **Google Drive** ou baixado diretamente

**Tempo estimado:** ~45–90 min (T4 gratuita)

---
## 📦 Célula 1 — Verificar GPU e instalar dependências

In [ ]:
# Verifica GPU disponível
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ GPU não encontrada — vá em Runtime > Change runtime type > T4 GPU')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Instala unsloth (instalação otimizada para Colab)
# unsloth: fine-tuning 2x mais rápido, 60% menos VRAM que HuggingFace puro
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
print('✅ Dependências instaladas!')

---
## 📂 Célula 2 — Montar Google Drive (opcional, para salvar o modelo)

In [ ]:
# Monte o Drive para salvar o modelo treinado (recomendado!)
# Se não quiser usar o Drive, pule esta célula e o modelo será salvo localmente

USE_DRIVE = True  # ← mude para False se não quiser usar o Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_SAVE_PATH = '/content/drive/MyDrive/medassist_model'
    print(f'✅ Drive montado. Modelo será salvo em: {MODEL_SAVE_PATH}')
else:
    MODEL_SAVE_PATH = '/content/medassist_model'
    print(f'⚠️  Sem Drive. Modelo salvo localmente em: {MODEL_SAVE_PATH}')
    print('   (será perdido ao reiniciar o Colab!)')

---
## 📊 Célula 3 — Upload e preparação do dataset

In [ ]:
# Opção A: Upload manual do arquivo
from google.colab import files
import os

DATASET_PATH = '/content/medquad_ptbr.jsonl'

if not os.path.exists(DATASET_PATH):
    print('📤 Faça o upload do arquivo medquad_ptbr.jsonl:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, DATASET_PATH)
    print(f'✅ Dataset carregado: {DATASET_PATH}')
else:
    print(f'✅ Dataset já existe: {DATASET_PATH}')

# Conta registros
with open(DATASET_PATH) as f:
    n = sum(1 for l in f if l.strip())
print(f'   Total de amostras: {n:,}')

In [ ]:
# Opção B: Carregar do Google Drive (se já tiver lá)
# Descomente se preferir esta opção

# DATASET_PATH = '/content/drive/MyDrive/medquad_ptbr.jsonl'
# print(f'Dataset: {DATASET_PATH}')

---
## 🔧 Célula 4 — Pré-processamento do dataset

In [ ]:
import json
import re
import random
from pathlib import Path

# ── Configurações ──────────────────────────────────────────
VAL_SPLIT    = 0.1     # 10% para validação
MAX_SAMPLES  = None    # None = usa tudo; coloque ex. 2000 para teste rápido
RANDOM_SEED  = 42
# ───────────────────────────────────────────────────────────

def clean_text(text):
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def load_and_process(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                msgs = r.get('messages', [])
                q = next((m['content'] for m in msgs if m['role'] == 'user'), '')
                a = next((m['content'] for m in msgs if m['role'] == 'assistant'), '')
                if len(q.strip()) >= 5 and len(a.strip()) >= 10:
                    records.append({
                        'question': clean_text(q),
                        'answer':   clean_text(a),
                    })
    return records

print('🔧 Carregando e processando dataset...')
all_records = load_and_process(DATASET_PATH)

if MAX_SAMPLES:
    random.seed(RANDOM_SEED)
    all_records = random.sample(all_records, min(MAX_SAMPLES, len(all_records)))
    print(f'   ⚠️  Limitado a {MAX_SAMPLES} amostras (modo rápido).')

random.seed(RANDOM_SEED)
random.shuffle(all_records)
n_val   = int(len(all_records) * VAL_SPLIT)
train   = all_records[n_val:]
val     = all_records[:n_val]

print(f'✅ Total válidos:  {len(all_records):,}')
print(f'   Treino:         {len(train):,}')
print(f'   Validação:      {len(val):,}')

# Mostra 2 exemplos
print('\n📋 Exemplo de amostra:')
ex = train[0]
print(f'  Pergunta: {ex["question"]}')
print(f'  Resposta: {ex["answer"][:200]}...')

---
## 🤖 Célula 5 — Carregar modelo base com Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Escolha do modelo ──────────────────────────────────────
# Opções recomendadas para T4 (15GB VRAM):
#   'unsloth/Phi-3-mini-4k-instruct'      ← 3.8B, RECOMENDADO, rápido
#   'unsloth/llama-3-8b-Instruct-bnb-4bit' ← 8B, mais capaz, ~60min
#   'unsloth/mistral-7b-instruct-v0.3-bnb-4bit' ← 7B, boa qualidade
#   'unsloth/gemma-2b-it-bnb-4bit'        ← 2B, muito rápido, qualidade menor

MODEL_NAME   = 'unsloth/Phi-3-mini-4k-instruct'  # ← ajuste aqui
MAX_SEQ_LEN  = 1024   # máximo de tokens por amostra
LOAD_IN_4BIT = True   # QLoRA: 4-bit para economizar VRAM
# ───────────────────────────────────────────────────────────

print(f'📦 Carregando {MODEL_NAME}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto-detecta (float16 na T4)
    load_in_4bit   = LOAD_IN_4BIT,
)
print(f'✅ Modelo carregado!')
print(f'   Parâmetros totais: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

---
## 🎯 Célula 6 — Configurar LoRA

In [ ]:
# ── Configurações LoRA ─────────────────────────────────────
LORA_R       = 16    # rank: 8-32 (maior = mais parâmetros treináveis)
LORA_ALPHA   = 32    # escala: geralmente 2x o rank
LORA_DROPOUT = 0.05
# ───────────────────────────────────────────────────────────

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # economiza VRAM
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA configurado!')
print(f'   Parâmetros treináveis: {trainable/1e6:.2f}M ({100*trainable/total:.2f}% do total)')
print(f'   Parâmetros congelados: {(total-trainable)/1e6:.0f}M')

---
## 📝 Célula 7 — Formatar dataset para fine-tuning

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = 'Você é um assistente médico especializado. Responda de forma precisa, clara e segura, sempre recomendando consulta médica profissional para decisões clínicas.'

def format_chat(record):
    """
    Formata no estilo ChatML / Instruct — compatível com Phi-3, LLaMA, Mistral.
    """
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': record['question']},
        {'role': 'assistant', 'content': record['answer']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

print('🔄 Formatando dataset...')
train_dataset = Dataset.from_list(train).map(format_chat)
val_dataset   = Dataset.from_list(val).map(format_chat)

print(f'✅ Datasets formatados!')
print(f'   Treino:    {len(train_dataset):,} amostras')
print(f'   Validação: {len(val_dataset):,} amostras')
print('\n📋 Exemplo formatado:')
print(train_dataset[0]['text'][:500] + '...')

---
## 🏋️ Célula 8 — Treinamento

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Hiperparâmetros de treinamento ─────────────────────────
# Perfil RÁPIDO (T4 free, ~45 min):  epochs=1, batch=4, grad_accum=4
# Perfil COMPLETO (T4 free, ~90 min): epochs=3, batch=2, grad_accum=8

NUM_EPOCHS          = 1       # ← aumente para 3 se tiver tempo/Colab Pro
BATCH_SIZE          = 4       # por device
GRAD_ACCUM_STEPS    = 4       # effective batch = 4*4 = 16
LEARNING_RATE       = 2e-4
WARMUP_RATIO        = 0.03
WEIGHT_DECAY        = 0.01
LOG_STEPS           = 25
SAVE_STEPS          = 200
MAX_STEPS           = -1      # -1 = roda todas as epochs; coloque ex. 100 para teste
# ───────────────────────────────────────────────────────────

training_args = TrainingArguments(
    per_device_train_batch_size   = BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM_STEPS,
    warmup_ratio                  = WARMUP_RATIO,
    num_train_epochs              = NUM_EPOCHS,
    max_steps                     = MAX_STEPS,
    learning_rate                 = LEARNING_RATE,
    fp16                          = not is_bfloat16_supported(),
    bf16                          = is_bfloat16_supported(),
    logging_steps                 = LOG_STEPS,
    optim                         = 'adamw_8bit',  # economiza VRAM
    weight_decay                  = WEIGHT_DECAY,
    lr_scheduler_type             = 'cosine',
    seed                          = 42,
    output_dir                    = '/content/checkpoints',
    save_steps                    = SAVE_STEPS,
    save_total_limit              = 2,
    evaluation_strategy           = 'steps',
    eval_steps                    = SAVE_STEPS,
    load_best_model_at_end        = True,
    report_to                     = 'none',
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc= 2,
    packing         = False,  # False = mais estável com dados médicos
    args            = training_args,
)

# Mostra uso de VRAM antes
gpu_stats = torch.cuda.get_device_properties(0)
used_mb   = round(torch.cuda.memory_reserved() / 1024**2)
total_mb  = round(gpu_stats.total_memory / 1024**2)
print(f'🖥️  GPU: {gpu_stats.name} | VRAM usada: {used_mb}MB / {total_mb}MB')
print(f'\n🚀 Iniciando treinamento...')
print(f'   Epochs: {NUM_EPOCHS} | Batch efetivo: {BATCH_SIZE*GRAD_ACCUM_STEPS}')
print(f'   Total de passos: ~{len(train_dataset)//(BATCH_SIZE*GRAD_ACCUM_STEPS)*NUM_EPOCHS}')

In [ ]:
# ▶️ EXECUTA O TREINAMENTO
trainer_stats = trainer.train()

# Mostra estatísticas
used_mb_after = round(torch.cuda.memory_reserved() / 1024**2)
print(f'\n✅ Treinamento concluído!')
print(f'   Tempo total:  {trainer_stats.metrics["train_runtime"]:.0f}s ({trainer_stats.metrics["train_runtime"]/60:.1f} min)')
print(f'   Train loss:   {trainer_stats.metrics["train_loss"]:.4f}')
print(f'   VRAM usada:   {used_mb_after}MB / {total_mb}MB')

---
## 📊 Célula 9 — Avaliação qualitativa (teste de respostas)

In [ ]:
from unsloth import FastLanguageModel

# Ativa modo inferência (mais rápido)
FastLanguageModel.for_inference(model)

DISCLAIMER = '\n\n⚠️ Esta resposta é informativa e não substitui avaliação médica profissional.'

def gerar_resposta(pergunta, max_tokens=400):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': pergunta},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = max_tokens,
        use_cache      = True,
        temperature    = 0.3,
        do_sample      = True,
        top_p          = 0.9,
    )
    # Decodifica apenas os novos tokens gerados
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response.strip() + DISCLAIMER

# Perguntas de teste
perguntas_teste = [
    'Quais são os sintomas da diabetes tipo 2?',
    'O que causa hipertensão arterial?',
    'Como funciona a quimioterapia?',
    'Quais são os fatores de risco para AVC?',
    'O que é leucemia linfoblástica aguda?',
]

print('💬 Testando o modelo fine-tuned:\n')
print('=' * 70)
for i, pergunta in enumerate(perguntas_teste, 1):
    print(f'\n[{i}] ❓ {pergunta}')
    resposta = gerar_resposta(pergunta)
    print(f'    💬 {resposta[:400]}')
    print('-' * 70)

---
## 💾 Célula 10 — Salvar modelo

In [ ]:
import os

print(f'💾 Salvando modelo em: {MODEL_SAVE_PATH}')
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# Salva adaptadores LoRA (pequeno, ~50–200MB)
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

# Opcional: merge LoRA no modelo base e salva em 16-bit
# (arquivo maior ~7GB, mas mais fácil de usar sem a lib PEFT)
MERGE_MODEL = False  # ← True para merge completo
if MERGE_MODEL:
    merged_path = MODEL_SAVE_PATH + '_merged'
    model.save_pretrained_merged(merged_path, tokenizer, save_method='merged_16bit')
    print(f'✅ Modelo merged salvo em: {merged_path}')

# Lista arquivos salvos
files_saved = list(os.listdir(MODEL_SAVE_PATH))
print(f'\n✅ Arquivos salvos ({len(files_saved)}):')
for f in sorted(files_saved):
    size = os.path.getsize(os.path.join(MODEL_SAVE_PATH, f)) / 1024**2
    print(f'   {f:<40} {size:.1f} MB')

---
## 📥 Célula 11 — Download dos adaptadores LoRA (se não usar Drive)

In [ ]:
# Compacta e baixa os adaptadores LoRA (~50–200MB)
# Use isso se não quiser usar o Google Drive

import shutil
from google.colab import files

ZIP_NAME = '/content/medassist_lora_adapters.zip'
shutil.make_archive(
    ZIP_NAME.replace('.zip', ''),
    'zip',
    MODEL_SAVE_PATH
)

size_mb = os.path.getsize(ZIP_NAME) / 1024**2
print(f'📦 ZIP criado: {ZIP_NAME} ({size_mb:.1f} MB)')
print('⬇️  Iniciando download...')
files.download(ZIP_NAME)

---
## 📊 Célula 12 — Métricas de treinamento

In [ ]:
import matplotlib.pyplot as plt

# Extrai histórico de loss
log_history = trainer.state.log_history
train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]

if train_losses:
    steps_train, losses_train = zip(*train_losses)
    steps_eval,  losses_eval  = zip(*eval_losses) if eval_losses else ([], [])

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps_train, losses_train, label='Train Loss', color='#2196F3', linewidth=2)
    if steps_eval:
        ax.plot(steps_eval, losses_eval, label='Val Loss', color='#F44336', linewidth=2, marker='o')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('MedAssist — Curva de Loss do Fine-tuning')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    plot_path = '/content/loss_curve.png'
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'📊 Gráfico salvo em: {plot_path}')

    # Salva no Drive também
    if USE_DRIVE:
        shutil.copy(plot_path, MODEL_SAVE_PATH + '/loss_curve.png')

    # Sumário final
    print(f'\n📈 Sumário do treinamento:')
    print(f'   Loss inicial:  {losses_train[0]:.4f}')
    print(f'   Loss final:    {losses_train[-1]:.4f}')
    print(f'   Redução:       {(1 - losses_train[-1]/losses_train[0])*100:.1f}%')
    if losses_eval:
        print(f'   Val loss final: {losses_eval[-1]:.4f}')
else:
    print('⚠️  Histórico de loss não disponível.')

---
## 🔄 Célula 13 — Como usar o modelo treinado no projeto local

Após baixar os adaptadores LoRA, integre ao projeto MedAssist assim:

In [ ]:
# Este bloco é apenas documentação — mostra como carregar o modelo localmente
# (não executa no Colab)

INSTRUCOES = '''
═══════════════════════════════════════════════════════════
  COMO USAR O MODELO TREINADO NO PROJETO LOCAL
═══════════════════════════════════════════════════════════

1. Baixe e descompacte os adaptadores LoRA (Célula 11)
   → Coloque a pasta em: medassist/outputs/model/

2. Atualize o config.py:
   BASE_MODEL_NAME = "unsloth/Phi-3-mini-4k-instruct"
   LORA_ADAPTERS_PATH = "outputs/model"

3. Para carregar com PEFT (sem GPU — CPU inference):

   from peft import PeftModel
   from transformers import AutoModelForCausalLM, AutoTokenizer

   base_model = AutoModelForCausalLM.from_pretrained(
       "microsoft/Phi-3-mini-4k-instruct",
       torch_dtype="auto",
       device_map="cpu",
   )
   model = PeftModel.from_pretrained(base_model, "outputs/model")
   tokenizer = AutoTokenizer.from_pretrained("outputs/model")

4. Ou use a API da Anthropic/OpenAI como LLM no LangChain
   (mais prático para CPU, veja chains.py)

═══════════════════════════════════════════════════════════
'''
print(INSTRUCOES)

---
## ✅ Checklist de conclusão

Após executar todas as células, verifique:

- [ ] GPU T4 ativa (Célula 1 mostra `CUDA disponível: True`)
- [ ] Dataset carregado com **5.274 amostras** (Célula 3)
- [ ] Treinamento concluído sem erros (Célula 8)
- [ ] Train loss decrescente (Célula 12 — gráfico)
- [ ] Respostas médicas coerentes em português (Célula 9)
- [ ] Modelo salvo no Drive ou baixado (Células 10/11)

---

**Próximo passo:** Integre os adaptadores LoRA ao projeto local (`medassist/outputs/model/`) e execute a Fase 3 (LangChain + LangGraph).